# S8.1 — PDF Upload Endpoint

Interactive verification of the PDF upload endpoint.

**What this spec does:**
- `POST /api/v1/upload` accepts PDF via multipart form
- Validates: extension, size (≤50MB), magic bytes
- Parses PDF with Docling → extracts sections, text, metadata
- Creates Paper record in PostgreSQL
- Chunks → embeds → indexes in OpenSearch
- Graceful degradation: paper saved even if indexing fails

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.upload.service import UploadService
from src.schemas.api.upload import UploadResponse
from src.schemas.pdf import PDFContent, Section

print("✓ Imports successful")

✓ Imports successful


## 1. UploadService — Validation

In [2]:
from io import BytesIO
from fastapi import UploadFile

service = UploadService(max_file_size_mb=50)

# Valid PDF
valid_pdf = UploadFile(file=BytesIO(b"%PDF-1.4 test content"), filename="test.pdf")
content = await service.validate_pdf(valid_pdf)
assert content.startswith(b"%PDF")
print(f"✓ Valid PDF accepted ({len(content)} bytes)")

# Non-PDF rejected
from src.exceptions import PDFValidationError
txt_file = UploadFile(file=BytesIO(b"not a pdf"), filename="paper.txt")
try:
    await service.validate_pdf(txt_file)
    assert False, "Should have raised"
except PDFValidationError as e:
    print(f"✓ Non-PDF rejected: {e.detail}")

# Empty file rejected
empty_file = UploadFile(file=BytesIO(b""), filename="empty.pdf")
try:
    await service.validate_pdf(empty_file)
    assert False, "Should have raised"
except PDFValidationError as e:
    print(f"✓ Empty file rejected: {e.detail}")

# Invalid magic bytes rejected
fake_pdf = UploadFile(file=BytesIO(b"FAKE content"), filename="fake.pdf")
try:
    await service.validate_pdf(fake_pdf)
    assert False, "Should have raised"
except PDFValidationError as e:
    print(f"✓ Invalid magic bytes rejected: {e.detail}")

✓ Valid PDF accepted (21 bytes)
✓ Non-PDF rejected: Only PDF files are accepted. Got: paper.txt
✓ Empty file rejected: Empty file uploaded
✓ Invalid magic bytes rejected: Invalid PDF file: fake.pdf does not have PDF magic bytes


## 2. Metadata Extraction

In [3]:
service = UploadService()

# With abstract section
content = PDFContent(
    raw_text="Full text of the paper",
    sections=[
        Section(title="Abstract", content="We present a novel approach.", level=1),
        Section(title="Introduction", content="Background info.", level=1),
    ],
)
meta = service._extract_metadata(content, "my_paper.pdf")
assert meta["abstract"] == "We present a novel approach."
assert meta["title"] == "Introduction"
print(f"✓ Title: {meta['title']}")
print(f"✓ Abstract: {meta['abstract']}")

# No abstract — falls back to raw_text
content2 = PDFContent(
    raw_text="This paper discusses neural networks.",
    sections=[Section(title="Methods", content="We use CNNs.", level=1)],
)
meta2 = service._extract_metadata(content2, "paper.pdf")
assert "neural networks" in meta2["abstract"]
print(f"✓ Fallback abstract: {meta2['abstract'][:50]}...")

# No sections — title from filename
content3 = PDFContent(raw_text="", sections=[])
meta3 = service._extract_metadata(content3, "research_paper_v2.pdf")
assert meta3["title"] == "research_paper_v2"
print(f"✓ Filename fallback title: {meta3['title']}")

✓ Title: Introduction
✓ Abstract: We present a novel approach.
✓ Fallback abstract: This paper discusses neural networks....
✓ Filename fallback title: research_paper_v2


## 3. UploadResponse Schema

In [4]:
import uuid

resp = UploadResponse(
    paper_id=uuid.uuid4(),
    arxiv_id="upload_abc123",
    title="Test Paper",
    authors=["Alice", "Bob"],
    abstract="A test abstract.",
    page_count=10,
    chunks_indexed=5,
    parsing_status="success",
    indexing_status="success",
    warnings=[],
    message="Paper uploaded successfully",
)

# Verify JSON serialization
data = resp.model_dump(mode="json")
assert "paper_id" in data
assert data["chunks_indexed"] == 5
assert data["parsing_status"] == "success"
print(f"✓ UploadResponse serializes correctly")
print(f"  paper_id: {data['paper_id']}")
print(f"  title: {data['title']}")
print(f"  chunks_indexed: {data['chunks_indexed']}")

✓ UploadResponse serializes correctly
  paper_id: 9a3f695c-bd1b-48f1-9463-23f6444afe27
  title: Test Paper
  chunks_indexed: 5


## 4. Full Endpoint Test (via ASGI transport)

In [5]:
from unittest.mock import AsyncMock, MagicMock
from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient
from src.routers.upload import router, get_pdf_parser, get_text_chunker, get_upload_service
from src.dependency import get_db_session, get_paper_repository, get_embeddings_client, get_opensearch_client

# Create mock services
mock_parser = AsyncMock()
mock_parser.parse_pdf = AsyncMock(return_value=PDFContent(
    raw_text="Full paper text about AI.",
    sections=[Section(title="Abstract", content="AI is great.", level=1)],
    page_count=5,
))

mock_repo = AsyncMock()
mock_paper = MagicMock()
mock_paper.id = uuid.uuid4()
mock_paper.arxiv_id = "upload_test"
mock_repo.create = AsyncMock(return_value=mock_paper)

mock_session = AsyncMock()
mock_chunker = MagicMock()
mock_chunker.chunk_paper = MagicMock(return_value=[])

mock_embeddings = AsyncMock()
mock_opensearch = MagicMock()

# Build app
app = FastAPI()
app.include_router(router, prefix="/api/v1")
app.dependency_overrides[get_db_session] = lambda: mock_session
app.dependency_overrides[get_paper_repository] = lambda: mock_repo
app.dependency_overrides[get_embeddings_client] = lambda: mock_embeddings
app.dependency_overrides[get_opensearch_client] = lambda: mock_opensearch
app.dependency_overrides[get_pdf_parser] = lambda: mock_parser
app.dependency_overrides[get_text_chunker] = lambda: mock_chunker
app.dependency_overrides[get_upload_service] = lambda: UploadService(max_file_size_mb=50)

async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    resp = await client.post(
        "/api/v1/upload",
        files={"file": ("test.pdf", BytesIO(b"%PDF-1.4 test"), "application/pdf")},
    )

assert resp.status_code == 200
data = resp.json()
print(f"✓ POST /api/v1/upload → {resp.status_code}")
print(f"  paper_id: {data['paper_id']}")
print(f"  parsing_status: {data['parsing_status']}")
print(f"  message: {data['message']}")

✓ POST /api/v1/upload → 200
  paper_id: a7e9cdfa-f4b5-462e-bf42-370ee31c79dd
  parsing_status: success
  message: Paper uploaded successfully: test (1 warning(s))


## 5. Router Registration in main.py

In [6]:
from src.main import create_app

app = create_app()
routes = [r.path for r in app.routes]
assert "/api/v1/upload" in routes
print(f"✓ Upload router registered at /api/v1/upload")
print(f"  Total routes: {len(routes)}")

✓ Upload router registered at /api/v1/upload
  Total routes: 13


In [7]:
print("\n" + "="*50)
print("S8.1 PDF Upload Endpoint — All checks passed!")
print("="*50)


S8.1 PDF Upload Endpoint — All checks passed!
